In [8]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import GridSearchCV, GroupKFold

df = pd.read_csv("training_set3.csv")

features = df[["ax", "ay", "az", "gx", "gy", "gz"]]
activity = df["label"]
groups = df["recording_id"]

# Create the decision tree
tree = DecisionTreeClassifier(random_state=42)

param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [None, 1, 2, 3, 4, 5], # Max depth of the tree
    'min_samples_leaf': [1, 2, 3, 4, 5] # Mininmum sample allowed in a leaf
}

group_cv = GroupKFold(n_splits=8)

grid = GridSearchCV(
    estimator=tree,
    param_grid=param_grid,
    cv=group_cv,
    scoring='accuracy',
    return_train_score=True,
    n_jobs=-1
)

grid.fit(features, activity, groups=groups)

# Best parameters
print("Best parameters:")
print(grid.best_params_)

# Best cross-validation accuracy
print("Best CV accuracy:")
print(grid.best_score_)

Best parameters:
{'criterion': 'gini', 'max_depth': None, 'min_samples_leaf': 5}
Best CV accuracy:
0.7798680926593897


In [9]:
# Investigate performance between CV folds
# Get the index of the best parameter combination
best_index = grid.best_index_

# Get the accuracy for each validation fold
fold_scores = []

for i in range(8):
    score = grid.cv_results_[f"split{i}_test_score"][best_index]
    fold_scores.append(score)

print("Fold accuracies:")

for i, score in enumerate(fold_scores, 1):
    print(f"Fold {i}: {score:.3f}")

print(f"\nMean accuracy: {sum(fold_scores) / len(fold_scores):.3f}")
print(f"Standard deviation: {pd.Series(fold_scores).std():.3f}")

Fold accuracies:
Fold 1: 0.779
Fold 2: 0.766
Fold 3: 0.825
Fold 4: 0.764
Fold 5: 0.732
Fold 6: 0.770
Fold 7: 0.773
Fold 8: 0.829

Mean accuracy: 0.780
Standard deviation: 0.032


In [10]:
results = pd.DataFrame(grid.cv_results_)
#results.columns

results[
    [
        'param_criterion',
        'param_min_samples_leaf',
        'param_max_depth',
        'mean_test_score',
        'std_test_score',
        'std_train_score',
        'mean_train_score'
    ]
].sort_values('mean_test_score', ascending=False)

,param_criterion,param_min_samples_leaf,param_max_depth,mean_test_score,std_test_score,std_train_score,mean_train_score
4,gini,5,None,0.779868,0.030324,0.001602,0.921380
34,entropy,5,None,0.779438,0.029565,0.001856,0.928005
33,entropy,4,None,0.776580,0.028558,0.001412,0.939297
3,gini,4,None,0.776331,0.029705,0.001287,0.931609
32,entropy,3,None,0.775520,0.028297,0.000922,0.955857
30,entropy,1,None,0.775028,0.028298,0.000000,1.000000
2,gini,3,None,0.772843,0.028877,0.001060,0.946938
31,entropy,2,None,0.771705,0.028702,0.000530,0.971520
0,gini,1,None,0.768697,0.027292,0.000000,1.000000
1,gini,2,None,0.767147,0.029600,0.000553,0.963903
